[//]: # (cr:doc name='chapter_4_column_deep_dive' id=e650f3a7)
# Chapter 4: Column Deep Dive

**Purpose:** Analyze each column in detail with distribution analysis, value validation, and transformation recommendations.

**What you'll learn:**
- How to validate value ranges for different column types
- How to interpret distribution shapes (skewness, kurtosis)
- When and why to apply transformations (log, sqrt, capping)
- How to detect zero-inflation and handle it

**Outputs:**
- Value range validation results
- Per-column distribution visualizations with statistics
- Skewness/kurtosis analysis with transformation recommendations
- Zero-inflation detection
- Type confirmation/override capability
- Updated exploration findings

[//]: # (cr:doc name='4_1_load_previous_findings' id=d8fea241)
## 4.1 Load Previous Findings

In [ ]:
# @cr:code name='init_progress' id=e30ac5f4
from customer_retention.analysis.notebook_progress import accept_workflow_params, track_and_export_previous

accept_workflow_params()
track_and_export_previous("04_column_deep_dive.ipynb")

import numpy as np
import plotly.graph_objects as go
from scipy import stats

from customer_retention.analysis.auto_explorer import ExplorationFindings, RecommendationRegistry
from customer_retention.analysis.visualization import (
    AttentionScorer,
    ChartBuilder,
    ColumnEntry,
    ColumnPaginator,
    console,
    display_figure,
    display_table,
)
from customer_retention.core.compat import native_pd, to_datetime
from customer_retention.core.config.column_config import ColumnType
from customer_retention.core.config.experiments import FINDINGS_DIR  # noqa: F401
from customer_retention.stages.profiling import (
    CategoricalDistributionAnalyzer,
    DistributionAnalyzer,
    TemporalAnalyzer,
    TemporalGranularity,
    TransformationType,
)
from customer_retention.stages.validation import DataValidator, RuleGenerator

In [ ]:
# @cr:code name='load_findings' id=113ca9a1
from customer_retention.analysis.auto_explorer import load_notebook_findings

FINDINGS_PATH, _namespace, dataset_name = load_notebook_findings(
    "04_column_deep_dive.ipynb", prefer_merged=True
)
print(f"Using: {FINDINGS_PATH}")

findings = ExplorationFindings.load(FINDINGS_PATH)
print(f"\nLoaded findings for {findings.column_count} columns from {findings.source_path}")

[//]: # (cr:doc name='4_2_load_source_data' id=05fa831d)
## 4.2 Load Source Data

In [ ]:
# @cr:code name='load_dataset' id=1f7af516
from customer_retention.analysis.auto_explorer.active_dataset_store import (
    require_silver_merged,
    require_silver_merged_distributed,
)
from customer_retention.analysis.auto_explorer.findings import classify_columns
from customer_retention.core.compat import is_databricks, track_stage_object
from customer_retention.stages.temporal import TEMPORAL_METADATA_COLS

df = require_silver_merged_distributed(_namespace) if is_databricks() else require_silver_merged(_namespace)
track_stage_object(df)
data_source = "silver_merged"

print(f"Loaded data from: {data_source}")
print(f"Shape: {df.shape}")

charts = ChartBuilder()
cc = classify_columns(findings, exclude=set(TEMPORAL_METADATA_COLS))
target_col = cc.target
entity_col = cc.identifier[0] if cc.identifier else None

# Initialize recommendation registry for this exploration
registry = RecommendationRegistry()
registry.init_bronze(findings.source_path)
if target_col:
    registry.init_gold(target_col)
if entity_col:
    registry.init_silver(entity_col)

# Merge all per-dataset recommendation files (collects NB02 bronze recs)
_per_dataset_recs = []
for _ds_name in _namespace.list_datasets():
    _ds_dir = _namespace.dataset_findings_dir(_ds_name)
    if _ds_dir.is_dir():
        for _rp in sorted(_ds_dir.glob("*_recommendations.yaml")):
            _per_dataset_recs.append(RecommendationRegistry.load(str(_rp)))
if _per_dataset_recs:
    registry = RecommendationRegistry.merge(_per_dataset_recs)
    if not registry.gold and target_col:
        registry.init_gold(target_col)
    if not registry.silver and entity_col:
        registry.init_silver(entity_col)

print(f"Initialized recommendation registry (Bronze: {findings.source_path})")

[//]: # (cr:doc name='4_3_value_range_validation' id=65fd32b1)
## 4.3 Value Range Validation

**📖 Interpretation Guide:**
- **Percentage fields** (rates): Should be 0-100 or 0-1 depending on format
- **Binary fields**: Should only contain 0 and 1
- **Count fields**: Should be non-negative integers
- **Amount fields**: Should be non-negative (unless refunds are possible)

**What to Watch For:**
- Rates > 100% suggest measurement or data entry errors
- Negative values in fields that should be positive
- Binary fields with values other than 0/1

**Actions:**
- Cap rates at 100 if they exceed (or investigate cause)
- Flag records with impossible negative values
- Convert binary fields to proper 0/1 encoding

In [ ]:
# @cr:code name='validate_ranges' id=c4ff22cb
validator = DataValidator()
range_rules = RuleGenerator.from_findings(findings)

console.start_section()
console.header("Value Range Validation")

if range_rules:
    range_results = validator.validate_value_ranges(df, range_rules)

    issues_found = []
    for r in range_results:
        detail = f"{r.invalid_values} invalid" if r.invalid_values > 0 else None
        console.check(f"{r.column_name} ({r.rule_type})", r.invalid_values == 0, detail)
        if r.invalid_values > 0:
            issues_found.append(r)

    all_invalid = sum(r.invalid_values for r in range_results)
    if all_invalid == 0:
        console.success("All value ranges valid")
    else:
        console.error(f"Found {all_invalid:,} values outside expected ranges")

        console.info("Examples of invalid values:")
        for r in issues_found[:3]:
            col = r.column_name
            if col in df.columns:
                if r.rule_type == 'binary':
                    invalid_mask = ~df[col].isin([0, 1, np.nan])
                    condition = "value not in [0, 1]"
                elif r.rule_type == 'non_negative':
                    invalid_mask = df[col] < 0
                    condition = "value < 0"
                elif r.rule_type == 'percentage':
                    invalid_mask = (df[col] < 0) | (df[col] > 100)
                    condition = "value < 0 or value > 100"
                elif r.rule_type == 'rate':
                    invalid_mask = (df[col] < 0) | (df[col] > 1)
                    condition = "value < 0 or value > 1"
                else:
                    continue

                invalid_values = df.loc[invalid_mask, col].dropna()
                if len(invalid_values) > 0:
                    examples = invalid_values.head(5).tolist()
                    console.metric(f"  {col}", f"{examples}")

                    # Add filtering recommendation
                    registry.add_bronze_filtering(
                        column=col, condition=condition, action="cap",
                        rationale=f"{r.invalid_values} values violate {r.rule_type} constraint",
                        source_notebook="04_column_deep_dive"
                    )

    console.info("Rules auto-generated from detected column types")
else:
    range_results = []
    console.info("No validation rules generated - no binary/numeric columns detected")

console.end_section()

[//]: # (cr:doc name='4_4_numeric_columns_analysis' id=7cf0f7be)
## 4.4 Numeric Columns Analysis

**📖 How to Interpret These Charts:**
- **Red dashed line** = Mean (sensitive to outliers)
- **Green solid line** = Median (robust to outliers)
- **Large gap between mean and median** = Skewed distribution
- **Long right tail** = Positive skew (common in count/amount data)

**📖 Understanding Distribution Metrics**

| Metric | Interpretation | Action |
|--------|---------------|--------|
| **Skewness** | Measures asymmetry | \|skew\| > 1: Consider log transform |
| **Kurtosis** | Measures tail heaviness | kurt > 10: Cap outliers before transform |
| **Zero %** | Percentage of zeros | > 40%: Use zero-inflation handling |

**📖 Transformation Decision Tree:**
1. If zeros > 40% → Create binary indicator + log(non-zeros)
2. If \|skewness\| > 1 AND kurtosis > 10 → Cap then log
3. If \|skewness\| > 1 → Log transform
4. If kurtosis > 10 → Cap outliers only
5. Otherwise → Standard scaling is sufficient

In [ ]:
# @cr:code name='analyze_distributions' id=a217318a
from customer_retention.core.compat.bulk_profiling import bulk_histogram

analyzer = DistributionAnalyzer()

numeric_cols = cc.numeric

# --- Phase 1: Bulk analysis (all columns) ---
analyses = analyzer.analyze_dataframe(df, numeric_cols)
recommendations = {col: analyzer.recommend_transformation(analysis)
                   for col, analysis in analyses.items()}

# --- Phase 2: Registry population (all columns) ---
for col_name in numeric_cols:
    rec = recommendations.get(col_name)
    if rec and rec.recommended_transform != TransformationType.NONE and registry.gold:
        registry.add_gold_transformation(
            column=col_name,
            transform=rec.recommended_transform.value,
            parameters=rec.parameters,
            rationale=rec.reason,
            source_notebook="04_column_deep_dive"
        )

transform_count = sum(1 for r in recommendations.values() if r and r.recommended_transform != TransformationType.NONE)
if transform_count > 0 and registry.gold:
    print(f"Added {transform_count} transformation recommendations to Gold layer")

# --- Phase 3: Paginated display ---
numeric_entries = []
for col in numeric_cols:
    analysis = analyses.get(col)
    if analysis:
        rec = recommendations.get(col)
        numeric_entries.append(ColumnEntry(
            name=col,
            column_type="numeric",
            attention_score=AttentionScorer.score_numeric(analysis, rec),
            analysis_data=analysis,
            recommendation_data=rec,
            extra={"col_info": findings.columns[col]},
        ))


def render_numeric(entry, output):
    analysis = entry.analysis_data
    rec = entry.recommendation_data
    col_name = entry.name
    col_info = entry.extra.get("col_info")

    print(f"\n{'='*70}")
    print(f"Column: {col_name} [score: {entry.attention_score.score:.0f} — {entry.attention_score.priority}]")
    if col_info:
        print(f"Type: {col_info.inferred_type.value} (Confidence: {col_info.confidence:.0%})")
    print("-" * 70)

    if analysis:
        print("\U0001f4ca Distribution Statistics:")
        print(f"   Mean: {analysis.mean:.3f}  |  Median: {analysis.median:.3f}  |  Std: {analysis.std:.3f}")
        print(f"   Range: [{analysis.min_value:.3f}, {analysis.max_value:.3f}]")
        if analysis.percentiles:
            print(f"   Percentiles: 1%={analysis.percentiles.get('p1', 0):.3f}, 25%={analysis.q1:.3f}, 75%={analysis.q3:.3f}, 99%={analysis.percentiles.get('p99', 0):.3f}")
        print("\n\U0001f4c8 Shape Analysis:")
        skew_label = '(Right-skewed)' if analysis.skewness > 0.5 else '(Left-skewed)' if analysis.skewness < -0.5 else '(Symmetric)'
        print(f"   Skewness: {analysis.skewness:.2f} {skew_label}")
        kurt_label = '(Heavy tails/outliers)' if analysis.kurtosis > 3 else '(Light tails)'
        print(f"   Kurtosis: {analysis.kurtosis:.2f} {kurt_label}")
        print(f"   Zeros: {analysis.zero_count:,} ({analysis.zero_percentage:.1f}%)")
        print(f"   Outliers (IQR): {analysis.outlier_count_iqr:,} ({analysis.outlier_percentage:.1f}%)")

        if rec:
            print(f"\n\U0001f527 Recommended Transformation: {rec.recommended_transform.value}")
            print(f"   Reason: {rec.reason}")
            print(f"   Priority: {rec.priority}")
            if rec.warnings:
                for warn in rec.warnings:
                    print(f"   \u26a0\ufe0f {warn}")

    hist = bulk_histogram(df, col_name, nbins=50)
    fig = go.Figure()

    if hist.counts:
        fig.add_trace(go.Bar(
            x=hist.bin_centers, y=hist.counts,
            width=[hist.bin_edges[i + 1] - hist.bin_edges[i] for i in range(len(hist.counts))],
            name='Distribution', marker_color='steelblue', opacity=0.7
        ))

    mean_val = analysis.mean
    median_val = analysis.median

    mean_position = "top right" if mean_val >= median_val else "top left"
    median_position = "top left" if mean_val >= median_val else "top right"

    fig.add_vline(
        x=mean_val, line_dash="dash", line_color="red",
        annotation_text=f"Mean: {mean_val:.2f}", annotation_position=mean_position,
        annotation_font_color="red", annotation_bgcolor="rgba(255,255,255,0.8)"
    )
    fig.add_vline(
        x=median_val, line_dash="solid", line_color="green",
        annotation_text=f"Median: {median_val:.2f}", annotation_position=median_position,
        annotation_font_color="green", annotation_bgcolor="rgba(255,255,255,0.8)"
    )

    if analysis and analysis.outlier_percentage > 5 and analysis.percentiles.get('p99') is not None:
        fig.add_vline(x=analysis.percentiles['p99'], line_dash="dot", line_color="orange",
                      annotation_text=f"99th: {analysis.percentiles['p99']:.2f}",
                      annotation_position="top right",
                      annotation_font_color="orange",
                      annotation_bgcolor="rgba(255,255,255,0.8)")

    transform_label = rec.recommended_transform.value if rec else "none"
    fig.update_layout(
        title=f"Distribution: {col_name}<br><sub>Skew: {analysis.skewness:.2f} | Kurt: {analysis.kurtosis:.2f} | Strategy: {transform_label}</sub>",
        xaxis_title=col_name, yaxis_title="Count",
        template='plotly_white', height=400
    )
    display_figure(fig)


def summary_numeric(entry):
    a = entry.analysis_data
    r = entry.recommendation_data
    return {
        "Column": entry.name,
        "Score": f"{entry.attention_score.score:.0f}",
        "Priority": entry.attention_score.priority,
        "Skewness": f"{a.skewness:.2f}",
        "Kurtosis": f"{a.kurtosis:.2f}",
        "Zeros %": f"{a.zero_percentage:.1f}%",
        "Outliers %": f"{a.outlier_percentage:.1f}%",
        "Transform": r.recommended_transform.value if r else "none",
    }


ColumnPaginator(
    entries=numeric_entries,
    render_callback=render_numeric,
    summary_callback=summary_numeric,
    title="Numeric Columns",
).show()

In [ ]:
# @cr:code name='display_distribution_stats' id=fa2eea04
if numeric_cols and analyses:
    stats_data = []
    for col_name in numeric_cols:
        analysis = analyses.get(col_name)
        if analysis and analysis.count > 0:
            stats_data.append({
                "feature": col_name,
                "count": analysis.count,
                "mean": analysis.mean,
                "std": analysis.std,
                "min": analysis.min_value,
                "25%": analysis.q1,
                "50%": analysis.median,
                "75%": analysis.q3,
                "95%": analysis.percentiles.get("p95", 0),
                "99%": analysis.percentiles.get("p99", 0),
                "max": analysis.max_value,
                "skewness": analysis.skewness,
                "kurtosis": analysis.kurtosis,
            })

    if stats_data:
        stats_df = native_pd.DataFrame(stats_data)

        display_stats = stats_df.copy()
        for col in ["mean", "std", "min", "25%", "50%", "75%", "95%", "99%", "max"]:
            display_stats[col] = display_stats[col].apply(lambda x: f"{x:.3f}")
        display_stats["skewness"] = display_stats["skewness"].apply(lambda x: f"{x:.3f}")
        display_stats["kurtosis"] = display_stats["kurtosis"].apply(lambda x: f"{x:.3f}")

        print("=" * 80)
        print("NUMERICAL FEATURE STATISTICS")
        print("=" * 80)
        display_table(display_stats)
    else:
        print("No numeric columns with data to display (all counts are 0)")

[//]: # (cr:doc name='4_5_distribution_summary_transformation_plan' id=71725704)
## 4.5 Distribution Summary & Transformation Plan

This table summarizes all numeric columns with their recommended transformations.

In [ ]:
# @cr:code name='build_transform_summary' id=742b062f
# Build transformation summary table (registry already populated above)
summary_data = []
for col_name in numeric_cols:
    analysis = analyses.get(col_name)
    rec = recommendations.get(col_name)

    if analysis and rec:
        summary_data.append({
            "Column": col_name,
            "Skewness": f"{analysis.skewness:.2f}",
            "Kurtosis": f"{analysis.kurtosis:.2f}",
            "Zeros %": f"{analysis.zero_percentage:.1f}%",
            "Outliers %": f"{analysis.outlier_percentage:.1f}%",
            "Transform": rec.recommended_transform.value,
            "Priority": rec.priority
        })

if summary_data:
    summary_df = native_pd.DataFrame(summary_data)
    display_table(summary_df)
else:
    console.info("No numeric columns to summarize")

[//]: # (cr:doc name='4_6_categorical_columns_analysis' id=172203af)
## 4.6 Categorical Columns Analysis

**📖 Distribution Metrics (Analogues to Numeric Skewness/Kurtosis):**

| Metric | Interpretation | Action |
|--------|---------------|--------|
| **Imbalance Ratio** | Largest / Smallest category count | > 10: Consider grouping rare categories |
| **Entropy** | Diversity measure (0 = one category, higher = more uniform) | Low entropy: May need stratified sampling |
| **Top-3 Concentration** | % of data in top 3 categories | > 90%: Rare categories may cause issues |
| **Rare Category %** | Categories with < 1% of data | High %: Group into "Other" category |

**📖 Encoding Recommendations:**
- **Low cardinality (≤5)** → One-hot encoding
- **Medium cardinality (6-20)** → One-hot or Target encoding
- **High cardinality (>20)** → Target encoding or Frequency encoding
- **Cyclical (days, months)** → Sin/Cos encoding

**⚠️ Common Issues:**
- Rare categories can cause overfitting with one-hot encoding
- High cardinality + one-hot = feature explosion
- Imbalanced categories may need special handling in train/test splits

In [ ]:
# @cr:code name='analyze_categorical' id=eb26a26a
cat_analyzer = CategoricalDistributionAnalyzer()

categorical_cols = cc.categorical

# --- Phase 1: Bulk analysis (all columns, single pass) ---
cat_analyses = cat_analyzer.analyze_dataframe(df, categorical_cols)

cyclical_cols = [c for c in cc.categorical
                 if findings.columns[c].inferred_type == ColumnType.CATEGORICAL_CYCLICAL]
cat_recommendations = cat_analyzer.get_all_recommendations(
    cyclical_columns=cyclical_cols, analyses=cat_analyses,
)

# --- Phase 2: Registry population (all columns) ---
for col_name, analysis in cat_analyses.items():
    rec = next((r for r in cat_recommendations if r.column_name == col_name), None)
    if rec and registry.gold:
        registry.add_gold_encoding(
            column=col_name,
            method=rec.encoding_type.value,
            rationale=rec.reason,
            source_notebook="04_column_deep_dive"
        )

if registry.gold:
    print(f"Added {len(cat_recommendations)} encoding recommendations to Gold layer")

# --- Phase 3: Paginated display ---
cat_entries = []
for col_name in categorical_cols:
    analysis = cat_analyses.get(col_name)
    if analysis:
        rec = next((r for r in cat_recommendations if r.column_name == col_name), None)
        cat_entries.append(ColumnEntry(
            name=col_name,
            column_type="categorical",
            attention_score=AttentionScorer.score_categorical(analysis, rec),
            analysis_data=analysis,
            recommendation_data=rec,
            extra={"col_info": findings.columns[col_name]},
        ))


def render_categorical(entry, output):
    analysis = entry.analysis_data
    rec = entry.recommendation_data
    col_name = entry.name
    col_info = entry.extra.get("col_info")

    print(f"\n{'='*70}")
    print(f"Column: {col_name} [score: {entry.attention_score.score:.0f} — {entry.attention_score.priority}]")
    if col_info:
        print(f"Type: {col_info.inferred_type.value} (Confidence: {col_info.confidence:.0%})")
    print("-" * 70)

    if analysis:
        print("\n Distribution Metrics:")
        print(f"   Categories: {analysis.category_count}")
        print(f"   Imbalance Ratio: {analysis.imbalance_ratio:.1f}x (largest/smallest)")
        print(f"   Entropy: {analysis.entropy:.2f} ({analysis.normalized_entropy*100:.0f}% of max)")
        print(f"   Top-1 Concentration: {analysis.top1_concentration:.1f}%")
        print(f"   Top-3 Concentration: {analysis.top3_concentration:.1f}%")
        print(f"   Rare Categories (<1%): {analysis.rare_category_count}")

        print("\n Interpretation:")
        if analysis.has_low_diversity:
            print("   LOW DIVERSITY: Distribution dominated by few categories")
        elif analysis.normalized_entropy > 0.9:
            print("   HIGH DIVERSITY: Categories are relatively balanced")
        else:
            print("   MODERATE DIVERSITY: Some category dominance but acceptable")

        if analysis.imbalance_ratio > 100:
            print("   SEVERE IMBALANCE: Rarest category has very few samples")
        elif analysis.is_imbalanced:
            print("   MODERATE IMBALANCE: Consider grouping rare categories")

        if rec:
            print("\n Recommendations:")
            print(f"   Encoding: {rec.encoding_type.value}")
            print(f"   Reason: {rec.reason}")
            print(f"   Priority: {rec.priority}")
            if rec.preprocessing_steps:
                print("   Preprocessing:")
                for step in rec.preprocessing_steps:
                    print(f"      - {step}")
            if rec.warnings:
                for warn in rec.warnings:
                    print(f"   Warning: {warn}")

    # Use pre-computed value_counts from analysis (no per-column Spark job)
    vc = analysis.value_counts if analysis else {}
    top_items = sorted(vc.items(), key=lambda x: x[1], reverse=True)[:10]
    subtitle = f"Entropy: {analysis.normalized_entropy*100:.0f}% | Imbalance: {analysis.imbalance_ratio:.1f}x | Rare: {analysis.rare_category_count}" if analysis else ""
    if top_items:
        fig = charts.bar_chart(
            [k for k, _ in top_items],
            [v for _, v in top_items],
            title=f"Top Categories: {col_name}<br><sub>{subtitle}</sub>"
        )
        display_figure(fig)


def summary_categorical(entry):
    a = entry.analysis_data
    r = entry.recommendation_data
    return {
        "Column": entry.name,
        "Score": f"{entry.attention_score.score:.0f}",
        "Priority": entry.attention_score.priority,
        "Categories": a.category_count,
        "Imbalance": f"{a.imbalance_ratio:.1f}x",
        "Entropy": f"{a.normalized_entropy*100:.0f}%",
        "Top-3 Conc.": f"{a.top3_concentration:.1f}%",
        "Rare (<1%)": a.rare_category_count,
        "Encoding": r.encoding_type.value if r else "N/A",
    }


ColumnPaginator(
    entries=cat_entries,
    render_callback=render_categorical,
    summary_callback=summary_categorical,
    title="Categorical Columns",
).show()

[//]: # (cr:doc name='4_7_datetime_columns_analysis' id=a12b9150)
## 4.7 Datetime Columns Analysis

**📖 Unlike numeric transformations, datetime analysis recommends NEW FEATURES to create:**

| Recommendation Type | Purpose | Examples |
|---------------------|---------|----------|
| **Feature Engineering** | Create predictive features from dates | `days_since_signup`, `tenure_years`, `month_sin_cos` |
| **Modeling Strategy** | How to structure train/test | Time-based splits when trends detected |
| **Data Quality** | Issues to address before modeling | Placeholder dates (1/1/1900) to filter |

**📖 Feature Engineering Strategies:**
- **Recency**: `days_since_X` - How recent was the event? (useful for predicting behavior)
- **Tenure**: `tenure_years` - How long has customer been active? (maturity/loyalty)
- **Duration**: `days_between_A_and_B` - Time between events (e.g., signup to first purchase)
- **Cyclical**: `month_sin`, `month_cos` - Preserves that December is near January
- **Categorical**: `is_weekend`, `is_quarter_end` - Behavioral indicators

In [ ]:
# @cr:code name='analyze_datetime_columns' id=569e111f
from customer_retention.core.compat import is_databricks
from customer_retention.core.compat.bulk_profiling import bulk_datetime_analysis_stats
from customer_retention.stages.profiling.temporal_analyzer import TemporalRecommendationType

datetime_cols = [c for c in cc.datetime if c in df.columns]

temporal_analyzer = TemporalAnalyzer()

# --- Pre-compute bulk stats (1-3 Spark jobs for all datetime columns) ---
dt_bulk = bulk_datetime_analysis_stats(df, datetime_cols) if datetime_cols else {}

# --- Phase 1: Per-column analysis ---
datetime_precomputed = {}
feature_engineering_recs = []
modeling_strategy_recs = []
data_quality_recs = []

for col_name in datetime_cols:
    other_dates = [c for c in datetime_cols if c != col_name]
    bulk = dt_bulk.get(col_name)

    if is_databricks() and bulk:
        analysis, growth, seasonality, recommendations = temporal_analyzer.analyze_all_from_bulk(
            bulk, col_name, other_date_columns=other_dates
        )
    else:
        date_series = to_datetime(df[col_name], errors='coerce', format='mixed')
        analysis = temporal_analyzer.analyze(date_series)
        growth = temporal_analyzer.calculate_growth_rate(date_series)
        seasonality = temporal_analyzer.analyze_seasonality(date_series)
        recommendations = temporal_analyzer.recommend_features(date_series, col_name, other_date_columns=other_dates)

    col_feature_recs = [r for r in recommendations if r.recommendation_type == TemporalRecommendationType.FEATURE_ENGINEERING]
    col_modeling_recs = [r for r in recommendations if r.recommendation_type == TemporalRecommendationType.MODELING_STRATEGY]
    col_quality_recs = [r for r in recommendations if r.recommendation_type == TemporalRecommendationType.DATA_QUALITY]

    feature_engineering_recs.extend(col_feature_recs)
    modeling_strategy_recs.extend(col_modeling_recs)
    data_quality_recs.extend(col_quality_recs)

    datetime_precomputed[col_name] = {
        "analysis": analysis,
        "growth": growth,
        "seasonality": seasonality,
        "feature_recs": col_feature_recs,
        "modeling_recs": col_modeling_recs,
        "quality_recs": col_quality_recs,
    }

# --- Phase 2: Registry population (all columns) ---
added_derived = 0
added_modeling = 0

if registry.silver:
    for rec in feature_engineering_recs:
        registry.add_silver_derived(
            column=rec.feature_name,
            expression=rec.code_hint or "",
            feature_type=rec.category,
            rationale=rec.reason,
            source_notebook="04_column_deep_dive"
        )
        added_derived += 1

seen_strategies = set()
for rec in modeling_strategy_recs:
    if rec.feature_name not in seen_strategies:
        registry.add_bronze_modeling_strategy(
            strategy=rec.feature_name,
            column=datetime_cols[0] if datetime_cols else "",
            parameters={"category": rec.category},
            rationale=rec.reason,
            source_notebook="04_column_deep_dive"
        )
        seen_strategies.add(rec.feature_name)
        added_modeling += 1

print(f"Added {added_derived} derived column recommendations to Silver layer")
print(f"Added {added_modeling} modeling strategy recommendations to Bronze layer")

# --- Phase 3: Paginated display ---
dt_entries = []
for col_name in datetime_cols:
    pre = datetime_precomputed.get(col_name)
    if pre:
        growth = pre["growth"]
        seasonality = pre["seasonality"]
        score_summary = {
            "column_name": col_name,
            "quality_issue_count": len(pre["quality_recs"]),
            "has_seasonality": seasonality.has_seasonality,
            "overall_growth_pct": growth.get("overall_growth_pct", 0) if growth.get("has_data") else 0,
            "feature_rec_count": len(pre["feature_recs"]),
            "modeling_rec_count": len(pre["modeling_recs"]),
        }
        dt_entries.append(ColumnEntry(
            name=col_name,
            column_type="datetime",
            attention_score=AttentionScorer.score_datetime(score_summary),
            analysis_data=pre,
            extra={"col_info": findings.columns[col_name]},
        ))


def render_datetime(entry, output):
    col_name = entry.name
    pre = entry.analysis_data
    col_info = entry.extra.get("col_info")
    analysis = pre["analysis"]
    growth = pre["growth"]
    seasonality = pre["seasonality"]
    col_feature_recs = pre["feature_recs"]
    col_modeling_recs = pre["modeling_recs"]
    col_quality_recs = pre["quality_recs"]

    # Use bulk-precomputed stats (no Spark job)
    bulk = dt_bulk.get(col_name)

    print(f"\n{'='*70}")
    print(f"Column: {col_name} [score: {entry.attention_score.score:.0f} — {entry.attention_score.priority}]")
    if col_info:
        print(f"Type: {col_info.inferred_type.value} (Confidence: {col_info.confidence:.0%})")
    print(f"{'='*70}")

    if bulk and bulk.min_date is not None:
        print(f"\n Date Range: {bulk.min_date} to {bulk.max_date}")
        print(f"   Nulls: {bulk.null_count:,} ({bulk.null_count / bulk.total_count * 100:.1f}%)" if bulk.total_count > 0 else "   Nulls: 0")
    print(f"   Auto-detected granularity: {analysis.granularity.value}")
    print(f"   Span: {analysis.span_days:,} days ({analysis.span_days/365:.1f} years)")

    if growth.get("has_data"):
        print("\n Growth Analysis:")
        print(f"   Trend: {growth['trend_direction'].upper()}")
        print(f"   Overall growth: {growth['overall_growth_pct']:+.1f}%")
        print(f"   Avg monthly growth: {growth['avg_monthly_growth']:+.1f}%")

    if seasonality.has_seasonality:
        print("\n Seasonality Detected:")
        print(f"   Peak months: {', '.join(seasonality.peak_periods[:3])}")
        print(f"   Trough months: {', '.join(seasonality.trough_periods[:3])}")
        print(f"   Seasonal strength: {seasonality.seasonal_strength:.2f}")

    if col_feature_recs:
        print("\n FEATURES TO CREATE:")
        for rec in col_feature_recs:
            priority_icon = "[HIGH]" if rec.priority == "high" else "[MED]" if rec.priority == "medium" else "[OK]"
            print(f"   {priority_icon} {rec.feature_name} ({rec.category})")
            print(f"      Why: {rec.reason}")
            if rec.code_hint:
                print(f"      Code: {rec.code_hint}")

    if col_modeling_recs:
        print("\n MODELING CONSIDERATIONS:")
        for rec in col_modeling_recs:
            priority_icon = "[HIGH]" if rec.priority == "high" else "[MED]" if rec.priority == "medium" else "[OK]"
            print(f"   {priority_icon} {rec.feature_name}")
            print(f"      Why: {rec.reason}")

    if col_quality_recs:
        print("\n DATA QUALITY ISSUES:")
        for rec in col_quality_recs:
            priority_icon = "[HIGH]" if rec.priority == "high" else "[MED]" if rec.priority == "medium" else "[OK]"
            print(f"   {priority_icon} {rec.feature_name}")
            print(f"      Why: {rec.reason}")
            if rec.code_hint:
                print(f"      Code: {rec.code_hint}")

    print("\n   Standard extractions available: year, month, day, day_of_week, quarter")

    # === VISUALIZATIONS (use pre-computed analysis — no Spark jobs) ===
    if growth.get("has_data"):
        fig = charts.growth_summary_indicators(growth, title=f"Growth Summary: {col_name}")
        display_figure(fig)

    chart_type = "line" if analysis.granularity in [TemporalGranularity.DAY, TemporalGranularity.WEEK] else "bar"
    fig = charts.temporal_distribution(analysis, title=f"Records Over Time: {col_name}", chart_type=chart_type)
    display_figure(fig)

    fig = charts.temporal_trend(analysis, title=f"Trend Analysis: {col_name}")
    display_figure(fig)

    if growth.get("has_data"):
        fig = charts.cumulative_growth_chart(growth["cumulative"], title=f"Cumulative Records: {col_name}")
        display_figure(fig)

    # Day-of-week from bulk pre-computed stats (no Spark job)
    if bulk and any(c > 0 for c in bulk.dow_counts):
        dow_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
        fig = go.Figure(go.Bar(x=dow_names, y=bulk.dow_counts, marker_color='steelblue'))
        fig.update_layout(title=f"Day of Week Distribution: {col_name}", xaxis_title="Day of Week", yaxis_title="Count", template='plotly_white')
        display_figure(fig)


def summary_datetime(entry):
    pre = entry.analysis_data
    analysis = pre["analysis"]
    growth = pre["growth"]
    seasonality = pre["seasonality"]
    return {
        "Column": entry.name,
        "Score": f"{entry.attention_score.score:.0f}",
        "Priority": entry.attention_score.priority,
        "Span (days)": analysis.span_days,
        "Seasonality": "Yes" if seasonality.has_seasonality else "No",
        "Trend": growth.get('trend_direction', 'N/A').capitalize() if growth.get("has_data") else "N/A",
        "Features": len(pre["feature_recs"]),
        "Quality Issues": len(pre["quality_recs"]),
    }


ColumnPaginator(
    entries=dt_entries,
    render_callback=render_datetime,
    summary_callback=summary_datetime,
    title="Datetime Columns",
).show()

# === DATETIME CROSS-COLUMN SUMMARY ===
if datetime_precomputed:
    print("\n" + "=" * 70)
    print("ALL RECOMMENDATIONS BY TYPE:")
    print("=" * 70)

    if feature_engineering_recs:
        print(f"\n FEATURES TO CREATE ({len(feature_engineering_recs)}):")
        for i, rec in enumerate(feature_engineering_recs, 1):
            priority_icon = "[HIGH]" if rec.priority == "high" else "[MED]" if rec.priority == "medium" else "[OK]"
            print(f"   {i}. {priority_icon} {rec.feature_name}")

    if modeling_strategy_recs:
        print(f"\n MODELING CONSIDERATIONS ({len(modeling_strategy_recs)}):")
        for i, rec in enumerate(modeling_strategy_recs, 1):
            priority_icon = "[HIGH]" if rec.priority == "high" else "[MED]" if rec.priority == "medium" else "[OK]"
            print(f"   {i}. {priority_icon} {rec.feature_name}: {rec.reason}")

    if data_quality_recs:
        print(f"\n DATA QUALITY TO ADDRESS ({len(data_quality_recs)}):")
        for i, rec in enumerate(data_quality_recs, 1):
            priority_icon = "[HIGH]" if rec.priority == "high" else "[MED]" if rec.priority == "medium" else "[OK]"
            print(f"   {i}. {priority_icon} {rec.feature_name}: {rec.reason}")


[//]: # (cr:doc name='4_8_type_override_optional' id=30b042f4)
## 4.8 Type Override (Optional)

If any column types were incorrectly inferred, you can override them here.

**Common overrides:**
- Binary columns detected as numeric → `ColumnType.BINARY`
- IDs detected as numeric → `ColumnType.IDENTIFIER`
- Ordinal categories detected as nominal → `ColumnType.CATEGORICAL_ORDINAL`

In [ ]:
# @cr:code name='apply_type_overrides' id=c144c829
# === TYPE OVERRIDES ===
# Uncomment and modify to override any incorrectly inferred types
TYPE_OVERRIDES = {
    # "column_name": ColumnType.NEW_TYPE,
    # Examples:
    # "is_active": ColumnType.BINARY,
    # "user_id": ColumnType.IDENTIFIER,
    # "satisfaction_level": ColumnType.CATEGORICAL_ORDINAL,
}

if TYPE_OVERRIDES:
    print("Applying type overrides:")
    for col_name, new_type in TYPE_OVERRIDES.items():
        if col_name in findings.columns:
            old_type = findings.columns[col_name].inferred_type.value
            findings.columns[col_name].inferred_type = new_type
            findings.columns[col_name].confidence = 1.0
            findings.columns[col_name].evidence.append("Manually overridden")
            print(f"  {col_name}: {old_type} → {new_type.value}")
else:
    print("No type overrides configured.")
    print("To override a type, add entries to TYPE_OVERRIDES dictionary above.")

[//]: # (cr:doc name='4_9_data_segmentation_analysis' id=1d0b0043)
## 4.9 Data Segmentation Analysis

**Purpose:** Determine if the dataset contains natural subgroups that might benefit from separate models.

**📖 Why This Matters:**
- Some datasets have distinct customer segments with very different behaviors
- A single model might struggle to capture patterns that vary significantly across segments
- Segmented models can improve accuracy but add maintenance complexity

**Recommendations:**
- **single_model** - Data is homogeneous; one model for all records
- **consider_segmentation** - Some variation exists; evaluate if complexity is worth it
- **strong_segmentation** - Distinct segments with different target rates; separate models likely beneficial

**Important:** This is exploratory guidance only. The final decision depends on business context, model complexity tolerance, and available resources.

In [ ]:
# @cr:code name='analyze_segments' id=fcecc407
from customer_retention.core.compat import is_databricks
from customer_retention.stages.profiling import SegmentAnalyzer, SparkSegmentAnalyzer

MAX_SEGMENT_SAMPLE_SIZE = 50_000

if is_databricks():
    segment_analyzer = SparkSegmentAnalyzer(max_sample_size=MAX_SEGMENT_SAMPLE_SIZE)
else:
    segment_analyzer = SegmentAnalyzer()

# Find target column if detected
target_col = None
for col_name, col_info in findings.columns.items():
    if col_info.inferred_type == ColumnType.TARGET:
        target_col = col_name
        break

# Run segmentation analysis using numeric features
print("="*70)
print("DATA SEGMENTATION ANALYSIS")
print("="*70)

segmentation = segment_analyzer.analyze(
    df,
    target_col=target_col,
    feature_cols=numeric_cols if numeric_cols else None,
    max_segments=5
)

print("\n🎯 Analysis Results:")
print(f"   Method: {segmentation.method.value}")
print(f"   Detected Segments: {segmentation.n_segments}")
print(f"   Cluster Quality Score: {segmentation.quality_score:.2f}")
if segmentation.target_variance_ratio is not None:
    print(f"   Target Variance Ratio: {segmentation.target_variance_ratio:.2f}")

print("\n📊 Segment Profiles:")
for profile in segmentation.profiles:
    target_info = f" | Target Rate: {profile.target_rate*100:.1f}%" if profile.target_rate is not None else ""
    print(f"   Segment {profile.segment_id}: {profile.size:,} records ({profile.size_pct:.1f}%){target_info}")

# Display recommendation card
fig = charts.segment_recommendation_card(segmentation)
display_figure(fig)

# Display segment overview
fig = charts.segment_overview(segmentation, title="Segment Overview")
display_figure(fig)

# Display feature comparison if we have features
if segmentation.n_segments > 1 and any(p.defining_features for p in segmentation.profiles):
    fig = charts.segment_feature_comparison(segmentation, title="Feature Comparison Across Segments")
    display_figure(fig)

print("\n📝 Rationale:")
for reason in segmentation.rationale:
    print(f"   • {reason}")

[//]: # (cr:doc name='4_10_save_updated_findings' id=52feca03)
## 4.10 Save Updated Findings

In [ ]:
# @cr:code name='save_findings' id=7b64fc2b
# Save updated findings back to the same file
findings.save(FINDINGS_PATH)
print(f"Updated findings saved to: {FINDINGS_PATH}")

# Save recommendations registry to merged path
recommendations_path = str(_namespace.merged_recommendations_path)
registry.save(recommendations_path)
print(f"Recommendations saved to: {recommendations_path}")

# Summary of recommendations
all_recs = registry.all_recommendations
print("\n\U0001f4cb Recommendations Summary:")
print(f"   Bronze layer: {len(registry.get_by_layer('bronze'))} recommendations")
print(f"   Silver layer: {len(registry.get_by_layer('silver'))} recommendations")
print(f"   Gold layer: {len(registry.get_by_layer('gold'))} recommendations")
print(f"   Total: {len(all_recs)} recommendations")

In [ ]:
# @cr:code name='release_stage_memory' id=517168ec
from customer_retention.core.compat import release_stage_memory

release_stage_memory()
del df, analyses, cat_analyses, datetime_precomputed, dt_bulk

[//]: # (cr:doc name='summary_what_we_learned' id=e1cde93c)
---

## Summary: What We Learned

In this notebook, we performed a deep dive analysis that included:

1. **Value Range Validation** - Validated rates, binary fields, and non-negative constraints
2. **Numeric Distribution Analysis** - Calculated skewness, kurtosis, and percentiles with transformation recommendations
3. **Categorical Distribution Analysis** - Calculated imbalance ratio, entropy, and concentration with encoding recommendations
4. **Datetime Analysis** - Analyzed seasonality, trends, and patterns with feature engineering recommendations
5. **Data Segmentation** - Evaluated if natural subgroups exist that might benefit from separate models

## Key Metrics Reference

**Numeric Columns:**
| Metric | Threshold | Action |
|--------|-----------|--------|
| Skewness | \|skew\| > 1 | Log transform |
| Kurtosis | > 10 | Cap outliers first |
| Zero % | > 40% | Zero-inflation handling |

**Categorical Columns:**
| Metric | Threshold | Action |
|--------|-----------|--------|
| Imbalance Ratio | > 10x | Group rare categories |
| Entropy | < 50% | Stratified sampling |
| Rare Categories | > 0 | Group into "Other" |

**Datetime Columns:**
| Finding | Action |
|---------|--------|
| Seasonality | Add cyclical month encoding |
| Strong trend | Time-based train/test split |
| Multiple dates | Calculate duration features |
| Placeholder dates | Filter or flag |

## Transformation & Encoding Summary

Review the summary tables above for:
- **Numeric**: Which columns need log transforms, capping, or zero-inflation handling
- **Categorical**: Which encoding to use and whether to group rare categories
- **Datetime**: Which temporal features to engineer based on detected patterns

---

## Next Steps

Continue to **02_source_integrity.ipynb** to:
- Analyze duplicate records and value conflicts
- Deep dive into missing value patterns
- Analyze outliers with IQR method
- Check data consistency
- Get cleaning recommendations

Or jump to **05_feature_opportunities.ipynb** if you want to see derived feature recommendations.

[//]: # (cr:doc name='section' id=418e3d9e)
> **Save Reminder:** Save this notebook (Ctrl+S / Cmd+S) before running the next one.
> The next notebook will automatically export this notebook's HTML documentation from the saved file.